# Compilation code 
Definition of variables and  paths for correct pipeline

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import random
import numpy as np
import cv2
import glob
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import Counter
from hailo_sdk_client import ClientRunner, InferenceContext

# Proyect  ID
MODEL_NAME = "yolov11n"
EXPERIMENT_TAG = "max_FPS"

# Path to files
BASE_DIR = ".."
PATHS = {
    "parsed_har": f"{BASE_DIR}/models/intermediate/{MODEL_NAME}_hailo_model.har",
    "calib_data": f"{BASE_DIR}/data/calib_dataset_full.npy",
    "val_imgs":   f"{BASE_DIR}/data/test/images",
    "val_labels": f"{BASE_DIR}/data/test/labels",
    "output_har": f"{BASE_DIR}/models/intermediate/{MODEL_NAME}_{EXPERIMENT_TAG}_quantized.har"
}

print(f"Loaded configuration for: {MODEL_NAME} [{EXPERIMENT_TAG}]")
print(f"Expected Output: {PATHS['output_har']}")

In [ ]:
def load_npy_calibration(npy_path):
    """Loads .npy dataset for calibration"""
    if not os.path.exists(npy_path):
        raise FileNotFoundError(f"File not found: {npy_path}")

    data = np.load(npy_path)
    print(f" Calibration dataset from .npy: {data.shape}")

    # Check type
    if data.dtype != np.float32:
        print(f"Converting data from {data.dtype} to float32")
        data = data.astype(np.float32)

    return data

def preprocess_image(img, target_shape):
    """Letterbox resize fot YOLO compatibility """
    h, w = img.shape[:2]
    target_h, target_w = target_shape
    scale = min(target_w / w, target_h / h)
    nw, nh = int(w * scale), int(h * scale)

    img_resized = cv2.resize(img, (nw, nh))
    image_padded = np.full((target_h, target_w, 3), 114, dtype=np.uint8)

    dy = (target_h - nh) // 2
    dx = (target_w - nw) // 2
    image_padded[dy:dy+nh, dx:dx+nw, :] = img_resized

    return image_padded, (h, w)

def load_yolo_labels(label_path, img_w, img_h):
    """Load YOLO labels and transform them to absolute  coordinates [x1, y1, x2, y2, cls]."""
    if not os.path.exists(label_path):
        return []
    boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            x_c, y_c, w_n, h_n = map(float, parts[1:])

            x1 = int((x_c - w_n/2) * img_w)
            y1 = int((y_c - h_n/2) * img_h)
            x2 = int((x_c + w_n/2) * img_w)
            y2 = int((y_c + h_n/2) * img_h)
            boxes.append([x1, y1, x2, y2, cls])
    return boxes

def box_iou(box1, box2):
    """IoU calculation."""
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    unionArea = box1Area + box2Area - interArea
    return interArea / unionArea if unionArea > 0 else 0


class YOLOv8PostProcess:
    def __init__(self, conf_thres=0.2, iou_thres=0.7):
        self.conf_thres = conf_thres
        self.iou_thres = iou_thres
        self.strides = [8, 16, 32]
        self.reg_max = 16
        self.project = np.arange(self.reg_max, dtype=np.float32)
        self.num_classes = None

    def softmax(self, x, axis=-1):
        x_max = np.max(x, axis=axis, keepdims=True)
        e_x = np.exp(x - x_max)
        return e_x / np.sum(e_x, axis=axis, keepdims=True)

    def decode_dfl(self, box_tensor):
        n, h, w, c = box_tensor.shape
        x = box_tensor.reshape(n, h, w, 4, self.reg_max)
        x = self.softmax(x, axis=-1)
        return np.dot(x, self.project)

    def make_anchors(self, feats, strides, grid_cell_offset=0.5):
        anchor_points, stride_tensor = [], []
        for i, stride in enumerate(strides):
            _, h, w, _ = feats[i].shape
            sx = np.arange(w, dtype=np.float32) + grid_cell_offset
            sy = np.arange(h, dtype=np.float32) + grid_cell_offset
            sy, sx = np.meshgrid(sy, sx, indexing='ij')
            anchor_points.append(np.stack((sx, sy), -1).reshape(-1, 2))
            stride_tensor.append(np.full((h * w, 1), stride, dtype=np.float32))
        return np.concatenate(anchor_points), np.concatenate(stride_tensor)

    def dist2bbox(self, distance, anchor_points, xywh=True):
        lt, rb = np.split(distance, 2, -1)
        x1y1 = anchor_points - lt
        x2y2 = anchor_points + rb
        if xywh:
            c_xy = (x1y1 + x2y2) / 2
            wh = x2y2 - x1y1
            return np.concatenate((c_xy, wh), -1)
        return np.concatenate((x1y1, x2y2), -1)

    def process(self, raw_outputs):
        feats_box = []
        feats_cls = []

        if not isinstance(raw_outputs, list) or len(raw_outputs) < 6:
            return np.empty((0, 6))

        for i in range(0, len(raw_outputs), 2):
            box = raw_outputs[i]
            cls = raw_outputs[i+1]
            if box.ndim == 3: box = box[np.newaxis, ...]
            if cls.ndim == 3: cls = cls[np.newaxis, ...]
            feats_box.append(box)
            feats_cls.append(cls)

        if self.num_classes is None:
            self.num_classes = feats_cls[0].shape[-1]

        anchors, strides = self.make_anchors(feats_cls, self.strides)
        pred_dist_list = []
        pred_scores_list = []

        for i, (box, cls) in enumerate(zip(feats_box, feats_cls)):
            cls = 1 / (1 + np.exp(-cls)) # Sigmoid
            b, h, w, c = box.shape

            box = self.decode_dfl(box).reshape(b, -1, 4)
            cls = cls.reshape(b, -1, self.num_classes)

            pred_dist_list.append(box)
            pred_scores_list.append(cls)

        pred_dist = np.concatenate(pred_dist_list, axis=1)
        pred_scores = np.concatenate(pred_scores_list, axis=1)

        boxes_cxcywh_grid = self.dist2bbox(pred_dist[0], anchors, xywh=True)
        boxes_cxcywh = boxes_cxcywh_grid * strides

        scores = pred_scores[0]

        class_ids = np.argmax(scores, axis=1)
        confidences = np.max(scores, axis=1)

        mask = confidences > self.conf_thres
        boxes_cxcywh = boxes_cxcywh[mask]
        confidences = confidences[mask]
        class_ids = class_ids[mask]

        if len(confidences) == 0:
            return np.empty((0, 6))

        boxes_tlwh = boxes_cxcywh.copy()
        boxes_tlwh[:, 0] = boxes_cxcywh[:, 0] - boxes_cxcywh[:, 2] / 2
        boxes_tlwh[:, 1] = boxes_cxcywh[:, 1] - boxes_cxcywh[:, 3] / 2

        indices = cv2.dnn.NMSBoxes(
            bboxes=boxes_tlwh.tolist(),
            scores=confidences.tolist(),
            score_threshold=self.conf_thres,
            nms_threshold=self.iou_thres
        )

        output = []
        if len(indices) > 0:
            indices = indices.flatten()
            for i in indices:
                b_tl = boxes_tlwh[i]
                x1, y1, w, h = b_tl
                x2, y2 = x1 + w, y1 + h
                output.append([x1, y1, x2, y2, confidences[i], class_ids[i]])

        return np.array(output)


def parse_hailo_nms(hailo_output, threshold=0.25):
    if isinstance(hailo_output, list): hailo_output = hailo_output[0]
    elif isinstance(hailo_output, dict): hailo_output = next(iter(hailo_output.values()))

    detections = []
    if hailo_output.ndim != 3: return np.empty((0, 6))

    num_classes, _, max_det = hailo_output.shape

    for c in range(num_classes):
        class_data = hailo_output[c]
        scores = class_data[4, :]
        valid = scores >= threshold
        if not np.any(valid): continue

        v_scores = scores[valid]
        v_coords = class_data[:4, valid]

        for k in range(len(v_scores)):
            ymin, xmin, ymax, xmax = v_coords[:, k]
            detections.append([xmin, ymin, xmax, ymax, v_scores[k], c])

    return np.array(detections) if detections else np.empty((0, 6))

def validate_full_dataset(runner, context_type, images_dir, labels_dir, num_classes=None):
    INPUT_SHAPE = (640, 640)
    CONF_THRESHOLD = 0.25
    IOU_THRESHOLD = 0.45
    BATCH_SIZE = 100

    native_decoder = YOLOv8PostProcess(conf_thres=CONF_THRESHOLD, iou_thres=IOU_THRESHOLD)

    image_files = sorted(glob.glob(os.path.join(images_dir, '*.jpg'))) + \
                  sorted(glob.glob(os.path.join(images_dir, '*.png')))
    if not image_files: return None

    # IMPORTANTE: Distinguir Native puro (raw) de FP Optimized (que suele ser como quant)
    is_raw_native = (context_type == InferenceContext.SDK_NATIVE)

    print(f"🔄 V7 Validando: {context_type} | Raw Native Mode: {is_raw_native}")

    processed_outputs = []
    all_gts = []
    all_orig_shapes = []

    for i in tqdm(range(0, len(image_files), BATCH_SIZE), desc="Inferencia"):
        batch_files = image_files[i : i + BATCH_SIZE]
        batch_imgs = []
        batch_shapes = []

        for img_path in batch_files:
            img = cv2.imread(img_path)
            if img is None: img = np.zeros((640, 640, 3), dtype=np.uint8)
            else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            processed, orig_shape = preprocess_image(img, INPUT_SHAPE)
            batch_imgs.append(processed)
            batch_shapes.append(orig_shape)

            label_name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
            gt = load_yolo_labels(os.path.join(labels_dir, label_name), orig_shape[1], orig_shape[0])
            all_gts.append(gt)
            all_orig_shapes.append(orig_shape)

        if not batch_imgs: continue
        batch_tensor = np.array(batch_imgs, dtype=np.uint8)

        with runner.infer_context(context_type) as ctx:
            if is_raw_native:
                input_data = batch_tensor.astype(np.float32) / 255.0
            else:
                input_data = batch_tensor

            raw_results = runner.infer(ctx, input_data)

        for j in range(len(batch_imgs)):
            if isinstance(raw_results, dict):
                img_output = {k: v[j] for k, v in raw_results.items()}
                if len(img_output) == 1: img_output = next(iter(img_output.values()))
            elif isinstance(raw_results, list):
                img_output = [layer[j] for layer in raw_results]
                if len(img_output) == 1: img_output = img_output[0]
            else:
                img_output = raw_results[j]

            if is_raw_native and isinstance(img_output, list) and len(img_output) >= 6:
                dets = native_decoder.process(img_output)
                processed_outputs.append((dets, 'pixel_640'))
            else:
                dets = parse_hailo_nms(img_output, CONF_THRESHOLD)
                processed_outputs.append((dets, 'normalized'))

    print("\n Printing metrics...")
    total_tp, total_fp, total_fn = 0, 0, 0

    for i in range(len(processed_outputs)):
        detections, mode = processed_outputs[i]
        gt_boxes = all_gts[i]
        h_orig, w_orig = all_orig_shapes[i]

        preds = []
        if detections.shape[0] > 0:
            for det in detections:
                x1, y1, x2, y2, score, cls_id = det
                if mode == 'normalized': # 0-1
                    x1, x2 = x1 * w_orig, x2 * w_orig
                    y1, y2 = y1 * h_orig, y2 * h_orig
                elif mode == 'pixel_640': # 0-640
                    sx, sy = w_orig / 640.0, h_orig / 640.0
                    x1, x2 = x1 * sx, x2 * sx
                    y1, y2 = y1 * sy, y2 * sy

                preds.append([x1, y1, x2, y2, score, cls_id])

        preds.sort(key=lambda x: x[4], reverse=True)
        matched = [False] * len(gt_boxes)

        for p in preds:
            best_iou = 0; best_idx = -1
            for idx, g in enumerate(gt_boxes):
                if not matched[idx] and int(g[4]) == int(p[5]):
                    iou = box_iou(p[:4], g[:4])
                    if iou > best_iou: best_iou = iou; best_idx = idx

            if best_iou >= IOU_THRESHOLD:
                total_tp += 1; matched[best_idx] = True
            else:
                total_fp += 1
        total_fn += (len(gt_boxes) - sum(matched))

    epsilon = 1e-7
    precision = total_tp / (total_tp + total_fp + epsilon)
    recall = total_tp / (total_tp + total_fn + epsilon)
    f1 = 2 * (precision * recall) / (precision + recall + epsilon)

    return {"precision": precision, "recall": recall, "f1": f1, "tp": total_tp, "fp": total_fp, "fn": total_fn}

def compare_hailo_modes(native_metrics=None, fp_metrics=None, q_metrics=None):
    """
    Compares  up to 3  models:
      1. NATIVE: Original outputs of parser (Raw Float32).
      2. FP_OPT: Output with preprocessing en Float32.
      3. QUANT:  Final ourput in Int8.
    """

    GREEN = "\033[92m"
    RED = "\033[91m"
    YELLOW = "\033[93m"
    BLUE = "\033[94m"
    RESET = "\033[0m"

    modes = []
    if native_metrics: modes.append(('NATIVE', native_metrics))
    if fp_metrics:     modes.append(('FP_OPT', fp_metrics))
    if q_metrics:      modes.append(('QUANT', q_metrics))

    if len(modes) < 2:
        print("❌ Error: Necesitas al menos 2 métricas para comparar.")
        return

    base_name, base_m = modes[0]
    final_name, final_m = modes[-1]

    def fmt(val): return f"{val:.2%}"
    def fmt_cnt(val): return f"{int(val)}"

    def fmt_delta(val, is_pct=True, inverse=False):
        txt = f"{val:+.2%}" if is_pct else f"{val:+d}"
        if abs(val) < (0.005 if is_pct else 1): return f"{RESET}{txt}{RESET}"
        good = val > 0 if not inverse else val < 0
        return f"{GREEN}{txt}{RESET}" if good else f"{RED}{txt}{RESET}"

    headers = [name for name, _ in modes] + [f"Δ ({base_name}->{final_name})"]
    col_width = 15
    row_fmt = "{:<15} | " + " | ".join([f"{{:<{col_width}}}" for _ in headers])

    print("\n" + "="*95)
    print(f" MULTI MODE COMPARATIVE: {BLUE}{' vs '.join([m[0] for m in modes])}{RESET}")
    print("="*95)
    print(row_fmt.format("METRICS", *headers))
    print("-" * 95)

    metrics_list = ['precision', 'recall', 'f1']
    for m in metrics_list:
        vals = [f"{mode[1][m]:.2%}" for mode in modes]
        delta = final_m[m] - base_m[m]
        print(row_fmt.format(m.capitalize(), *vals, fmt_delta(delta)))

    print("-" * 95)

    counts_list = [('tp', False), ('fp', True), ('fn', True)] # (metric, is_inverse)
    for m, is_inv in counts_list:
        vals = [f"{mode[1][m]}" for mode in modes]
        delta = final_m[m] - base_m[m]
        print(row_fmt.format(m.upper(), *vals, fmt_delta(delta, False, is_inv)))

    print("="*95)
    print(f"\n {YELLOW}PIPELINE DIAGNOSTICS:{RESET}")

    if native_metrics and fp_metrics:
        delta_pre = fp_metrics['f1'] - native_metrics['f1']
        if abs(delta_pre) > 0.02:
             print(f"   {RED}PREPROCESS  ALERT:{RESET} Big change  detected ({delta_pre:+.2%}) when aplying model script.")
        else:
             print(f"   {GREEN}Correct pre-processing:{RESET} Model script configuration mantains original metrics.")

    ref_metrics = fp_metrics if fp_metrics else native_metrics
    ref_name = "FP_OPT" if fp_metrics else "NATIVE"

    if q_metrics and ref_metrics:
        delta_quant = q_metrics['f1'] - ref_metrics['f1']

        if abs(delta_quant) < 0.01:
             print(f"   {GREEN}PERFECT CUANTIZATION: {RESET} Minimal  loss ({delta_quant:+.2%}).")
        elif abs(delta_quant) < 0.03:
             print(f"   {YELLOW}ACCEPTABLE QUANTIZATION:{RESET} Moderated loss ({delta_quant:+.2%}).")
        else:
             print(f"   {RED}QUANTIZATION FAILURE:{RESET} Considerable loss ({delta_quant:+.2%}).")
             if q_metrics['recall'] < ref_metrics['recall']:
                 print("      - Couse: Recall loss (Objects dissapear).")

    print("="*95 + "\n")


## Optimization strategies (Model Scripts)
This  function generates (`.alls`) dinamically, depending on selected strategy.

### yolov8n

In [ ]:
def get_model_script(strategy="balanced"):
    """
    Generates Hailo Runner commands based on a specific strategy.
    Strategies:
    'baseline': Only normalization and NMS (fast, for pipeline testing).
    'balanced': Level 2 optimization + Compression (good balance between FPS/Accuracy).
    'accuracy': Keeps critical layers in high precision (16-bit).
    """

    # Basic commands
    base_commands = [
        "normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])",
        f"nms_postprocess('../tools/NMS/nms_layer_config_{MODEL_NAME}.json',meta_arch=yolov8, engine=cpu)\n"
    ]

    # Specific strategies
    strategies = {
        "baseline": [
            "model_optimization_flavor(optimization_level=0)"
        ],

        "balanced": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "change_output_activation(conv42, sigmoid)",
            "change_output_activation(conv53, sigmoid)",
            "change_output_activation(conv63, sigmoid)",
            "allocator_param(width_splitter_defuse=disabled)"
        ],

        "accuracy": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "model_optimization_flavor(optimization_level=2, compression_level=0)",
            "performance_param(compiler_optimization_level=max)",
            "quantization_param([conv80, conv65, conv54], precision_mode=a16_w16)",
            "model_optimization_config(calibration, calibset_size=1500)",
            "pre_quantization_optimization(activation_clipping, layers={*}, mode=percentile, clipping_values=[0.01, 99.99])"
        ]
    }

    if strategy not in strategies:
        raise ValueError(f"Estrategia desconocida: {strategy}. Usa: {list(strategies.keys())}")


    full_script = base_commands + strategies[strategy]
    return "\n".join(full_script)

### yolov11n

In [ ]:
def get_model_script(strategy="balanced"):
    """
    Generates Hailo Runner commands based on a specific strategy.
    Strategies:
    'baseline': Only normalization and NMS (fast, for pipeline testing).
    'balanced': Level 2 optimization + Compression (good balance between FPS/Accuracy).
    'accuracy': Keeps critical layers in high precision (16-bit).
    """

    # Basic commands
    base_commands = [
        "normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])",
        f"nms_postprocess('../tools/NMS/nms_layer_config_{MODEL_NAME}.json',meta_arch=yolov8, engine=cpu)\n"
    ]

    # Specific strategies
    strategies = {
        "baseline": [
            "model_optimization_flavor(optimization_level=0)",
        ],

        "balanced": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "change_output_activation(conv54, sigmoid)",
            "change_output_activation(conv65, sigmoid)",
            "change_output_activation(conv80, sigmoid)",
            "allocator_param(width_splitter_defuse=disabled)"
        ],

        "accuracy": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "model_optimization_flavor(optimization_level=2, compression_level=1)",
            "performance_param(compiler_optimization_level=max)",
            "allocator_param(width_splitter_defuse=disabled)",
            "change_output_activation(conv54, sigmoid)",
            "change_output_activation(conv65, sigmoid)",
            "change_output_activation(conv80, sigmoid)",
            "quantization_param([conv1, conv4, conv54, conv65, conv80], precision_mode=a16_w16)",
            # Clipping de percentil para eliminar valores atípicos (outliers) antes de cuantizar
            "pre_quantization_optimization(activation_clipping, layers={*}, mode=percentile, clipping_values=[0.01, 99.99])"
        ],

        "max_FPS": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "model_optimization_flavor(optimization_level=3, compression_level=4)",
            "model_optimization_config(compression_params, auto_4bit_weights_ratio=0.3)",
            "resources_param(max_control_utilization=1.0, max_compute_utilization=1.0, max_memory_utilization=1.0)",
            "allocator_param(width_splitter_defuse=disabled)",
            "change_output_activation(conv54, sigmoid)",
            "change_output_activation(conv65, sigmoid)",
            "change_output_activation(conv80, sigmoid)",
            "model_optimization_config(calibration, calibset_size=1500)",
            # Clipping de percentil para eliminar valores atípicos (outliers) antes de cuantizar
            "pre_quantization_optimization(activation_clipping, layers={*}, mode=percentile, clipping_values=[0.01, 99.99])"
        ]


    }

    if strategy not in strategies:
        raise ValueError(f"Estrategia desconocida: {strategy}. Usa: {list(strategies.keys())}")

    full_script = base_commands + strategies[strategy]
    return "\n".join(full_script)

### yolov11s

In [ ]:
def get_model_script(strategy="balanced"):
    """
    Generates Hailo Runner commands based on a specific strategy.
    Strategies:
    'baseline': Only normalization and NMS (fast, for pipeline testing).
    'balanced': Level 2 optimization + Compression (good balance between FPS/Accuracy).
    'accuracy': Keeps critical layers in high precision (16-bit).
    """

    # Basic commands
    base_commands = [
        "normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])",
        f"nms_postprocess('../tools/NMS/nms_layer_config_{MODEL_NAME}.json',meta_arch=yolov8, engine=cpu)\n"
    ]

    # Specific strategies
    strategies = {
        "baseline": [
            "model_optimization_flavor(optimization_level=0)",
        ],

        "balanced": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "change_output_activation(conv54, sigmoid)",
            "change_output_activation(conv65, sigmoid)",
            "change_output_activation(conv80, sigmoid)",
        ],

        "accuracy": [
            "model_optimization_config(checker_cfg, policy=enabled, analyze_mode=advanced)",
            "model_optimization_flavor(optimization_level=2, compression_level=0)",
            "performance_param(compiler_optimization_level=max)",
            "quantization_param([conv80, conv65, conv54], precision_mode=a16_w16)",
            "model_optimization_config(calibration, calibset_size=1500)",
            "pre_quantization_optimization(activation_clipping, layers={*}, mode=percentile, clipping_values=[0.01, 99.99])"
        ]
    }

    if strategy not in strategies:
        raise ValueError(f"Estrategia desconocida: {strategy}. Usa: {list(strategies.keys())}")

    full_script = base_commands + strategies[strategy]
    return "\n".join(full_script)

## Pipeline execution
Loads model, applies script and converts to intermediate .har format

In [ ]:
#Strategy selection
SELECTED_STRATEGY = "max_FPS"

print("-" * 60)
print(f" Validating model: {MODEL_NAME} (Strategy: {EXPERIMENT_TAG})")
print("-" * 60)


# Loads model and  model script
runner = ClientRunner(har=PATHS['parsed_har'])
print("Loaded model.")

script_commands = get_model_script(SELECTED_STRATEGY)
runner.load_model_script(script_commands)
print(f" Script '{SELECTED_STRATEGY}' applied.")


# Native (Float32)

print("\n[1/3] Calculating native (Raw Float32)...")
native_metrics = validate_full_dataset(
    runner,
    InferenceContext.SDK_NATIVE,
    PATHS['val_imgs'],
    PATHS['val_labels'],
    num_classes=2
)

# Optimization full precision (FP32 -> FP Optimized)
print("\n Executing optimize_full_precision()...")
runner.optimize_full_precision()

print("[2/3] Calculating Baseline FP Optimized (Float32 + Norm)...")
fp_metrics = validate_full_dataset(
    runner,
    InferenceContext.SDK_FP_OPTIMIZED,
    PATHS['val_imgs'],
    PATHS['val_labels'],
    num_classes=2
)

# Quantization (INT8)
print("\n Loading calibration data and quantizising...")
calib_data = load_npy_calibration(PATHS['calib_data'])
runner.optimize(calib_data)

print("[3/3] Calculating quantize model (Int8)...")
q_metrics = validate_full_dataset(
    runner,
    InferenceContext.SDK_QUANTIZED,
    PATHS['val_imgs'],
    PATHS['val_labels'],
    num_classes=2
)

# Results
compare_hailo_modes(
    native_metrics=native_metrics,
    fp_metrics=fp_metrics,
    q_metrics=q_metrics
)

#Save converted model
runner.save_har(PATHS['output_har'])
print(f"💾 Modelo final guardado en: {PATHS['output_har']}")